# Задача 15. Оптимальные параметры вычисления интеграла

Исследуется интеграл функции `sin(x)` на отрезке `[-100; 100]`. Данные получены программой `task15` с использованием `Stopwatch`, каждый вариант измерен три раза.

In [ ]:
#r "nuget:ScottPlot, 5.1.59"

using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;

Formatter.Register(typeof(ScottPlot.Plot), (plot, writer) =>
    writer.Write(((ScottPlot.Plot)plot).GetPngHtml(1000, 650)),
    HtmlFormatter.MimeType);

In [ ]:
var currentDirectory = Directory.GetCurrentDirectory();
var resultsDirectory = Directory.Exists(Path.Combine(currentDirectory, "results"))
    ? Path.Combine(currentDirectory, "results")
    : Path.Combine(currentDirectory, "task15", "results");
var csvPath = Path.Combine(resultsDirectory, "thread-benchmarks.csv");

if (!File.Exists(csvPath))
    throw new FileNotFoundException("Не найден файл результатов", csvPath);

var measurements = File.ReadLines(csvPath)
    .Skip(1)
    .Where(line => !string.IsNullOrWhiteSpace(line))
    .Select(line => line.Split(','))
    .Select(parts => new
    {
        Threads = int.Parse(parts[0], CultureInfo.InvariantCulture),
        Result = double.Parse(parts[1], CultureInfo.InvariantCulture),
        AverageMilliseconds = double.Parse(parts[^1], CultureInfo.InvariantCulture)
    })
    .ToArray();

if (measurements.Length != 16)
    throw new InvalidOperationException("Ожидались замеры для 16 вариантов числа потоков");
if (measurements.Any(item => Math.Abs(item.Result) > 1e-4))
    throw new InvalidOperationException("Требуемая точность не достигнута");

var optimal = measurements.MinBy(item => item.AverageMilliseconds);
optimal

In [ ]:
double[] timeValues = measurements.Select(item => item.AverageMilliseconds).ToArray();
double[] threadValues = measurements.Select(item => (double)item.Threads).ToArray();

ScottPlot.Plot performancePlot = new();
performancePlot.Add.Scatter(timeValues, threadValues);
performancePlot.Title("Solve performance by thread count");
performancePlot.XLabel("Average time, ms");
performancePlot.YLabel("Thread count");
performancePlot

In [ ]:
var optimalResultPath = Path.Combine(resultsDirectory, "optimal-result.txt");
var optimalResultText = File.ReadAllText(optimalResultPath);

if (!optimalResultText.Contains("Требование ускорения не менее 15%: выполнено."))
    throw new InvalidOperationException("Требование ускорения не подтверждено");

optimalResultText

## Итог

Минимальное среднее время получено при 15 потоках. Многопоточная версия быстрее настоящей однопоточной реализации на 88,21%, поэтому требование преимущества не менее 15% выполнено.